In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
import pandas as pd

In [13]:
cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

fleet_path = (BASE_DIR / "data" / "processed" / "modeling_dataset.csv")
fleet_df = pd.read_csv(fleet_path)

#print columns
print(fleet_df.columns)

Index(['date', 'amb_temp_max_c', 'amb_temp_min_c', 'amb_temp_avg_c',
       'humidity', 'wind_gust_kph', 'blower_id', 'site_id',
       'operational_class', 'max_op_ambient_temp_c', 'failure_threshold',
       'max_op_rpm', 'daily_op_hours', 'cumulative_op_hours',
       'daily_load_percent', 'rpm', 'dust_index', 'pressure_diff_psi',
       'casing_temperature_c', 'vibration_mm_s', 'daily_degradation',
       'degradation_index', 'maintenance_event', 'health_score',
       'maintenance_required', 'failure_event', 'RUL_days',
       'maintenance_due_14d', 'maintenance_due_30d',
       'previous_maintenance_event', 'maintenance_cycle',
       'days_since_maintenance', 'cycle_start_cumulative_hours',
       'hours_since_maintenance'],
      dtype='str')


In [14]:
oem_interval_hours = 6000
oem_interval_days = 365

In [15]:
fleet_df["date"] = pd.to_datetime(fleet_df["date"])
fleet_df = fleet_df.sort_values(by=["blower_id", "date"]).copy()

In [ ]:
#creating OEM maintenance trigger
fleet_df["oem_maintenance_due"] = (
    (fleet_df["hours_since_maintenance"] >= oem_interval_hours) |
    (fleet_df["days_since_maintenance"] >= oem_interval_days)
).astype(int)

fleet_df["oem_maintenance_due"].value_counts()

fleet_df["predictive_maintenance_due"] = fleet_df["maintenance_due_30d"]
fleet_df["predictive_maintenance_due"].value_counts()

SyntaxError: invalid syntax (3092976763.py, line 2)

In [ ]:
#comparing maintenance due counts
comparison_counts = pd.DataFrame({
    "policy": ["OEM", "Predictive"],
    "maintenance_due": [fleet_df["oem_maintenance_due"].sum(),
     fleet_df["predictive_maintenance_due"].sum()]})

comparison_counts

#comparing average degradation when maintenance due
policy_comparison = pd.DataFrame({
    "policy": ["OEM", "Predictive"],
    "avg_degradation_when_due": [
        fleet_df.loc[fleet_df["oem_maintenance_due"] == 1, "degradation_index"].mean(),
        fleet_df.loc[fleet_df["predictive_maintenance_due"] == 1, "degradation_index"].mean()
    ],
    "avg_rul_when_due": [
        fleet_df.loc[fleet_df["oem_maintenance_due"] == 1, "RUL_days"].mean(),
        fleet_df.loc[fleet_df["predictive_maintenance_due"] == 1, "RUL_days"].mean()
    ]})

policy_comparison

#Comparing high risk days
high_risk_threshold = 0.70

fleet_df["high_risk_day"] = (fleet_df["degradation_index"] >= high_risk_threshold).astype(int)
high_risk_comparison = pd.DataFrame({
    "policy": ["OEM", "Predictive"],
    "high_risk_days_captured": [
        fleet_df.loc[fleet_df["oem_maintenance_due"] == 1, "high_risk_day"].sum(),
        fleet_df.loc[fleet_df["predictive_maintenance_due"] == 1, "high_risk_day"].sum()
    ]})
    
high_risk_comparison